## Laboratorio 07 - Problema de CartPole con Deep Q-Learning 

In [4]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

### Creación de Entorno CartPole

In [3]:
env = gym.make('CartPole-v1')

print("¡Entorno CartPole creado exitosamente!\n")
print(f"Espacio de observación: {env.observation_space}")
print(f"Espacio de acción: {env.action_space}")
print(f"Número de acciones posibles: {env.action_space.n}")

state, info = env.reset()
print(f"Dimensiones del estado: {env.observation_space.shape}")
print(f"Estado inicial de ejemplo: {state}")

¡Entorno CartPole creado exitosamente!

Espacio de observación: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Espacio de acción: Discrete(2)
Número de acciones posibles: 2
Dimensiones del estado: (4,)
Estado inicial de ejemplo: [ 0.03386014 -0.00959698  0.00293532 -0.02388402]


### Definición de las redes en línea y de destino

In [6]:
class DQN(nn.Module):
    """Red neuronal para Deep Q-Learning"""
    def __init__(self, state_size=4, action_size=2):
        super(DQN, self).__init__()
        
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64) 
        self.fc3 = nn.Linear(64, action_size)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
# Creación de 2 redes neuronales
online_net = DQN(state_size=4, action_size=2)  # Red en línea para selección de acciones
target_net = DQN(state_size=4, action_size=2)  # Red de destino para estimación de valores Q

In [8]:
# Red destino con mismos pesos que la red en línea (Solo inicialmente)
target_net.load_state_dict(online_net.state_dict())

<All keys matched successfully>

In [9]:
print("Redes neuronales creadas exitosamente!")
print(f"Red en línea: {online_net}")
print(f"Red de destino: {target_net}")
print("La red de destino ha sido inicializada con los mismos pesos que la red en línea.")

Redes neuronales creadas exitosamente!
Red en línea: DQN(
  (fc1): Linear(in_features=4, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
)
Red de destino: DQN(
  (fc1): Linear(in_features=4, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
)
La red de destino ha sido inicializada con los mismos pesos que la red en línea.


### Establecer Hiperparámetros

In [11]:
# Para el entrenamiento
NUM_EPISODES = 1000          
BATCH_SIZE = 32              

# Factor de descuento
GAMMA = 0.99                 

# Parámetros de exploración (epsilon-greedy)
EPSILON = 1.0                # 100%
EPSILON_DECAY = 0.995      
EPSILON_MIN = 0.01           # 1% exploración mínima

# Adicionales para optimización
LEARNING_RATE = 0.001        # Tasa de aprendizaje
MEMORY_SIZE = 10000          # Tamaño del buffer de experiencia
TARGET_UPDATE_FREQ = 10      # Frecuencia de actualización de la red de destino (cada N episodios)